# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis and Time Windows

- **Grain:** One row represents one content item for one client, identified by `client_hash_id` and `content_hash_id`.
- **Feature window:** February 1, 2026 through February 28, 2026. The `prior_*` columns are calculated only from this earlier window.
- **Target window:** March 1, 2026 through March 31, 2026. `future_impressions` and `future_decline_label` are calculated only from this later window.

In [1]:
%pip -q install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

# Find the repository root so this works from the notebook folder.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)

print(f"Loaded local cache: {cache_path}")
print(f"Rows: {len(dataframe):,}")
print(f"Columns: {len(dataframe.columns)}")

Loaded local cache: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\february_march_features.parquet
Rows: 321,546
Columns: 9


In [3]:
# Verify the stated grain: each client-content pair should occur once.
grain_counts = (
    dataframe.groupby(['client_hash_id', 'content_hash_id'])
    .size()
    .reset_index(name='row_count')
)

duplicate_pairs = grain_counts[grain_counts['row_count'] > 1]

grain_results = pd.DataFrame({
    'total_rows': [len(dataframe)],
    'unique_client_content_pairs': [len(grain_counts)],
    'duplicate_pairs': [len(duplicate_pairs)],
    'unique_clients': [dataframe['client_hash_id'].nunique()],
    'unique_content_items': [dataframe['content_hash_id'].nunique()],
})

grain_results

,total_rows,unique_client_content_pairs,duplicate_pairs,unique_clients,unique_content_items
0,321546,321546,0,54,321546


The grain check compares the number of rows with the number of unique `client_hash_id` and `content_hash_id` pairs. A `duplicate_pairs` value of zero confirms that the cached dataset has exactly one row per client-content pair.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features: known before the target window

These columns are calculated from February and are safe inputs for a March prediction:

- `prior_impressions`
- `prior_clicks`
- `prior_avg_position`
- `prior_sessions`
- `prior_engagement_rate`

### Label: the observed future outcome

- `future_decline_label`: 1 when March impressions are less than 80% of February impressions, for rows with at least 100 February impressions; 0 otherwise.
- `future_impressions`: the observed March impression total used to calculate the label. It is never a feature.

### Context: used for joining, grouping, and splitting

- `client_hash_id`: used for client-grouped train/test splits.
- `content_hash_id`: used to identify content items and inspect results.

### Excluded

- `future_impressions`: future information and part of the label definition.
- `future_decline_label`: the target, never a feature.
- `client_hash_id` and `content_hash_id`: identifiers, not learned signals.
- Any March metric: it would leak information from the target period into the features.
- Raw daily rows: already aggregated into the February and March windows before this file was created.

### Missing-value policy

- Missing `prior_avg_position` means there were no February impressions available to calculate position.
- Missing `prior_sessions` or `prior_engagement_rate` means the February GA4 denominator was unavailable or zero.
- Do not blindly replace these values with zero without recording a missingness flag or choosing an explicit model imputation strategy.

## 3. Verify it with checks

*Every contract claim gets a check next to it.*

*The checks below use the local parquet cache created from the huggingface warehouse, so the warehouse is not queried again*

### 1. Grain verification

This check proves that one row equals one `client_hash_id` x `content_hash_id` pair.

In [38]:
grain_results

,total_rows,unique_client_content_pairs,duplicate_pairs,unique_clients,unique_content_items
0,321546,321546,0,54,321546


### 2. Missingness and target verification

The cached file does not contain daily dates because the windows had already been aggregated previously. This check verifies the resulting columns, missingness, eligibility rule, and observed target rate.

In [39]:
expected_columns = [
    'client_hash_id',
    'content_hash_id',
    'prior_impressions',
    'prior_clicks',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
    'future_impressions',
    'future_decline_label',
]

missing_columns = sorted(set(expected_columns) - set(dataframe.columns))
missingness = dataframe[expected_columns].isna().mean().sort_values(ascending=False)
eligible_rows = dataframe['future_decline_label'].notna()

contract_check = pd.DataFrame({
    'total_rows': [len(dataframe)],
    'eligible_rows': [eligible_rows.sum()],
    'ineligible_rows': [(~eligible_rows).sum()],
    'future_decline_rate': [dataframe.loc[eligible_rows, 'future_decline_label'].mean()],
    'missing_columns': [', '.join(missing_columns) if missing_columns else 'none'],
})

print('Missingness by column:')
print(missingness)
print('\nContract summary:')
contract_check

Missingness by column:
prior_engagement_rate    0.896292
future_decline_label     0.750201
prior_avg_position       0.522435
prior_sessions           0.465492
client_hash_id           0.000000
prior_clicks             0.000000
prior_impressions        0.000000
content_hash_id          0.000000
future_impressions       0.000000
dtype: float64

Contract summary:


,total_rows,eligible_rows,ineligible_rows,future_decline_rate,missing_columns
0,321546,80322,241224,0.217549,none


### 3. Feature, label, and context check

This check confirms that the feature columns are February `prior` fields, that the future fields are kept separate, and that the client/content identifiers are available for grouped splitting and auditing.

In [40]:
feature_columns = [
    'prior_impressions',
    'prior_clicks',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
]
label_columns = ['future_impressions', 'future_decline_label']
context_columns = ['client_hash_id', 'content_hash_id']

field_check = pd.DataFrame({
    'bucket': ['features', 'labels', 'context'],
    'columns': [feature_columns, label_columns, context_columns],
})

field_check

,bucket,columns
0,features,"[prior_impressions, prior_clicks, prior_avg_po..."
1,labels,"[future_impressions, future_decline_label]"
2,context,"[client_hash_id, content_hash_id]"


## 4. Data limits

- The local file contains February features and March outcomes only; it does not support a different prediction month without rebuilding the cache.
- A row with no February impressions cannot produce a reliable percentage-change label, so its target is null and it is excluded from evaluation.
- Missing GA4 values may reflect unavailable tracking rather than true zero engagement.
- Missing position means no measurable February search position, not rank zero.
- This is observational data. The label records an impression decline but does not prove that any feature caused it.
- The cache is generated data and is ignored by Git; it must be regenerated if deleted.

In [7]:
# Verify the cached file's windows and target rule as metadata checks.
window_check = pd.DataFrame({
    'feature_window': ['2026-02-01 to 2026-02-28'],
    'target_window': ['2026-03-01 to 2026-03-31'],
    'minimum_prior_impressions': [100],
    'decline_threshold': ['future_impressions < 0.8 * prior_impressions'],
    'target_is_null_below_threshold': [
        dataframe.loc[dataframe['prior_impressions'] < 100, 'future_decline_label'].isna().all()
    ],
})

window_check

,feature_window,target_window,minimum_prior_impressions,decline_threshold,target_is_null_below_threshold
0,2026-02-01 to 2026-02-28,2026-03-01 to 2026-03-31,100,future_impressions < 0.8 * prior_impressions,True


## Self-check

- [x] The grain is one client-content pair.
- [x] February features and March outcomes are separated.
- [x] Features, labels, context, and excluded fields are documented.
- [x] The grain, missingness, eligibility, and window checks run on the local cache.
- [x] No private client names, URLs, or raw queries are included.
- [x] The claims are observational and decision-support only.